# 00 — Environment and configuration tour

Milestone **M0**. This notebook checks the environment and explores `Qwen3Config`.
It does not build a model and does not download anything.

In [ ]:
from qwen3_scratch.env import collect_environment

collect_environment()

## Qwen3-0.6B-Base hyperparameters

Note that `head_dim` (128) is **not** `hidden_size / num_attention_heads` (64):
the query projection maps 1024 → 2048 features.

In [ ]:
from qwen3_scratch.config import QWEN3_0_6B_BASE as cfg

print(f"q_proj: {cfg.hidden_size} -> {cfg.q_proj_dim}")
print(f"k_proj/v_proj: {cfg.hidden_size} -> {cfg.kv_proj_dim}")
print(f"GQA group size: {cfg.num_queries_per_kv}")
print(f"parameters: {cfg.estimate_num_parameters():,}")
print(f"non-embedding parameters: {cfg.estimate_num_parameters(include_embeddings=False):,}")

## Where do the parameters live?

Per decoder layer: attention projections, QK-Norm weights, SwiGLU MLP and two RMSNorms.

In [ ]:
h, d_ff = cfg.hidden_size, cfg.intermediate_size
per_layer = {
    "attention (q,k,v,o)": h * cfg.q_proj_dim + 2 * h * cfg.kv_proj_dim + cfg.q_proj_dim * h,
    "qk_norm": 2 * cfg.head_dim,
    "mlp (gate,up,down)": 3 * h * d_ff,
    "block RMSNorms": 2 * h,
}
for name, n in per_layer.items():
    print(f"{name:<22}{n:>12,}")
print(f"{'embedding (tied)':<22}{cfg.vocab_size * h:>12,}")